<h1 style="text-align:center"><b>[Step 2]</b></h1>
<h2 style="text-align:center"><b>Labeling Process</b></h2>
<p style="text-align:center">Combining LLMs with Gathered Surveys of Real Human Perspectives</p>

In [1]:
import pandas as pd
import numpy as np
from transformers import pipeline
import re
import time
import os
import torch
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

c:\Users\justi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\justi\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
pip install tf-keras


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os, torch
from transformers import pipeline

print("torch:", torch.__version__)
print("TF blocked:", os.environ.get("TRANSFORMERS_NO_TF"))


torch: 2.7.1+cu118
TF blocked: 1


### Enable GPU

In [4]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import torch

# Check if CUDA is available
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

# If available, set device to GPU
if torch.cuda.is_available():
    device = torch.device("cuda:0")  # Use GPU 0
    print(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA not available, using CPU")

CUDA available: True
Number of GPUs: 1
Using device: NVIDIA GeForce RTX 2060


### Disable GPU

In [6]:
os.environ["CUDA_VISIBLE_DEVICES"] = ""

# *[I] Feature Extractor Class*

In [7]:
class FakeReviewFeatureExtractor:
    
    def __init__(self):
        # Contradiction patterns
        self.contradiction_patterns = [
            "belum coba", "belum dicoba", "belum test", "belum tes",
            "belum pakai", "belum dipakai", "belum pake", "belum dipake",
            "blm coba", "blm cb", "blm tes", "blm test",
            "blm pakai", "blm dipakai", "blm pake", "blm dipake",
            "semoga cocok", "semoga awet", "semoga tahan lama"
        ]
        
        # Template indicators
        self.template_indicators = [
            "produk sangat bagus", "pengiriman sangat cepat",
            "packing super aman", "penjual fast respon",
            "recommended banget", "recomended banget",
            "harga terjangkau", "harga bersahabat",
            "kualitas original", "produk original"
        ]
        
        # Excessive emotion patterns (REGEX SAFE)
        self.excessive_emotion_patterns = {
            "caps": r'[A-Z]{5,}',
            "hyperbolic": r'(woy|anjir|astaga|tolonggg)',
            "exclamation": r'[!]{2,}',
            "emoji": r'[\U0001F300-\U0001FAFF]{3,}'  # unicode emoji range
        }
        
        # Promotional language
        self.promo_patterns = [
            "wajib beli", "wajib dibeli", "harus beli", "harus dibeli",
            "best seller", "top seller", "fast respon", "amanah terus"
        ]
        
        # Gibberish / spam
        self.gibberish_patterns = [
            r'(.)\1{4,}',
            r'([a-z]{1,3})\1{3,}',
            r'^[a-z]{1,5}$',
            r'\b(wkwk|wkkw|hehe|hihi|huhu|xixi){3,}\b'
        ]
    
    def extract_features(self, text):
        text_lower = text.lower()
        words = text.split()
        word_count = len(words)

        template_count = sum(p in text_lower for p in self.template_indicators)

        features = {
            # Contradiction
            "has_contradiction": int(any(p in text_lower for p in self.contradiction_patterns)),
            "contradiction_count": sum(p in text_lower for p in self.contradiction_patterns),

            # Template
            "template_count": template_count,
            "template_density": template_count / max(word_count, 1),

            # Emotional excess
            "excessive_caps": int(bool(re.search(self.excessive_emotion_patterns["caps"], text))),
            "hyperbolic_words": int(bool(re.search(self.excessive_emotion_patterns["hyperbolic"], text_lower))),
            "multiple_exclamation": int(bool(re.search(self.excessive_emotion_patterns["exclamation"], text))),
            "emoji_overuse": int(bool(re.search(self.excessive_emotion_patterns["emoji"], text))),

            # Promotional
            "promo_indicators": sum(p in text_lower for p in self.promo_patterns),

            # Gibberish / spam
            "has_gibberish": int(any(re.search(p, text_lower) for p in self.gibberish_patterns)),
            "char_repetition_ratio": self._calculate_repetition_ratio(text_lower),
            "is_spam_engagement": int(self._is_spam_engagement(text_lower)),

            # Length
            "word_count": word_count,
            "is_too_short": int(word_count <= 5),
            "is_too_long": int(word_count >= 100),
            "avg_word_length": np.mean([len(w) for w in words]) if words else 0,

            # Structure
            "has_ellipsis": int("..." in text or "…" in text),
            "punctuation_density": sum(c in "!?.,;:" for c in text) / max(len(text), 1),
            "newline_count": text.count("\n"),

            # Semantics
            "mentions_original": int("original" in text_lower or "ori" in text_lower),
            "mentions_fast_delivery": int(any(p in text_lower for p in ["cepat banget", "sampai cepat", "pengiriman cepat"])),
            "rating_mismatch_keywords": int(any(p in text_lower for p in ["kurang", "kecewa", "tidak sesuai", "mengecewakan"]))
        }

        # Suspicion score (weighted)
        features["total_suspicion_score"] = (
            features["has_contradiction"] * 3 +
            features["template_count"] * 2 +
            features["promo_indicators"] * 1.5 +
            features["excessive_caps"] * 1.5 +
            features["is_too_short"] * 1 +
            features["has_gibberish"] * 5 +
            features["is_spam_engagement"] * 4
        )

        return features
    
    def _calculate_repetition_ratio(self, text):
        if len(text) < 5:
            return 0
        repeated = len(re.findall(r'(.)\1{2,}', text))
        return min(1.0, repeated / len(text))
    
    def _is_spam_engagement(self, text):
        text_no_space = text.replace(" ", "")
        if len(text_no_space) < 10:
            return False

        if len(re.findall(r'([a-z]{1,3})\1{2,}', text_no_space)) >= 3:
            return True

        vowels = sum(c in "aeiou" for c in text_no_space)
        if vowels / len(text_no_space) < 0.15:
            return True

        words = text.split()
        if len(words) == 1 and len(words[0]) > 15:
            if len(re.findall(r'[^aeiou]{4,}', words[0])) >= 2:
                return True

        return False

# *[II] Hybrid Detector Class*

In [8]:
class HybridFakeReviewDetector:
    
    def __init__(self):
        self.feature_extractor = FakeReviewFeatureExtractor()
        self.classifier = None  # Lazy loading
        
        self.thresholds = {
            'fake': 3.0,
            'high_confidence_real': -1.0
        }
    
    def _init_llm(self):
        """Initialize LLM only when needed"""
        if self.classifier is None:
            print("Loading LLM model (one-time initialization)...")
            self.classifier = pipeline(
                "zero-shot-classification",
                model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
                hypothesis_template="Ulasan ini {}.",
                device=-1
            )
            print("LLM loaded!")
    
    def detect(self, text, rating=None, helpful_count=None, use_llm=False):
        
        # Extract features
        features = self.feature_extractor.extract_features(text)
        
        llm_fake_score = 0.0
        if use_llm and features['total_suspicion_score'] > 1.5:
            self._init_llm()
            labels = [
                "adalah ulasan palsu dengan bahasa template atau kontradiksi",
                "adalah ulasan asli dari pengalaman nyata pembeli"
            ]
            llm_result = self.classifier(text[:512], labels)
            llm_fake_score = llm_result["scores"][0] if llm_result["labels"][0] == labels[0] else llm_result["scores"][1]
        
        # Rating mismatch
        rating_mismatch = 0
        if rating is not None:
            if rating >= 4 and features['rating_mismatch_keywords']:
                rating_mismatch = 2
            elif rating <= 2 and features['template_count'] > 2:
                rating_mismatch = 1
        
        # Final decision
        final_score = features['total_suspicion_score'] + (llm_fake_score * 3) + rating_mismatch
        
        if final_score >= self.thresholds['fake']:
            decision = "Fake"
            confidence = min(0.95, 0.6 + (final_score / 20))
        else:
            decision = "Real"
            confidence = max(0.55, 1 - (final_score / 10))


        
        return {
            'text': text[:100] + '...' if len(text) > 100 else text,
            'decision': decision,
            'confidence': round(confidence, 3),
            'suspicion_score': round(final_score, 2),
            'key_features': {
                'contradiction': features['has_contradiction'],
                'template_language': features['template_count'],
                'excessive_emotion': features['excessive_caps'] + features['emoji_overuse'],
                'gibberish_spam': features['has_gibberish'] or features['is_spam_engagement'],
                'word_count': features['word_count']
            },
            'llm_contribution': round(llm_fake_score, 3)
        }

print("Detector class created!")

Detector class created!


# *[III] Batch Processing Function*

In [9]:
def process_dataset(csv_path, sample_size=None, use_llm=False, save_every=1000):
    """
    Process dataset with progress tracking and checkpoints
    
    Parameters:
    - csv_path: Path to CSV file
    - sample_size: Limit number of reviews (None = all)
    - use_llm: False = fast (recommended), True = slow but slightly more accurate
    - save_every: Save checkpoint every N reviews
    """
    
    # Load data
    df = pd.read_csv(csv_path)
    
    if 'id' not in df.columns:
        df = df.reset_index(drop=True)
        df.insert(0, 'id', df.index + 1)  # id mulai dari 1
    
    if sample_size:
        df = df.sample(n=min(sample_size, len(df)), random_state=42)
    
    print(f"\n{'='*60}")
    print(f"Processing {len(df)} reviews...")
    print(f"Mode: {'LLM ENABLED (slow)' if use_llm else 'FAST MODE (rule-based only)'}")
    print(f"Expected time: {len(df) * (12 if use_llm else 0.01) / 60:.1f} minutes")
    print(f"{'='*60}\n")
    
    # Initialize detector
    detector = HybridFakeReviewDetector()
    
    # Process reviews
    results = []
    start_time = time.time()
    
    for idx, row in df.iterrows():
        if idx % 100 == 0 and idx > 0:
            elapsed = time.time() - start_time
            reviews_per_sec = idx / elapsed
            remaining = (len(df) - idx) / reviews_per_sec / 60
            print(f"✓ Processed {idx}/{len(df)} | Speed: {reviews_per_sec:.1f} reviews/sec | ETA: {remaining:.1f} min")
        
        result = detector.detect(
            text=row['ulasan_cleaned'],
            rating=row.get('Rating'),
            helpful_count=row.get('Membantu'),
            use_llm=use_llm
        )
        
        results.append({
            'id': row['id'],
            'original_text': row['ulasan_cleaned'],
            'rating': row.get('Rating'),
            'helpful': row.get('Membantu'),
            **result
        })
        
        # Checkpoint saves
        if len(results) % save_every == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(f'checkpoint_{len(results)}.csv', index=False)
            print(f"  💾 Checkpoint saved: {len(results)} reviews")
    
    results_df = pd.DataFrame(results)
    
    # Summary
    elapsed_total = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"✅ COMPLETED in {elapsed_total/60:.1f} minutes")
    print(f"   Speed: {len(df)/elapsed_total:.1f} reviews/second")
    print(f"{'='*60}")
    
    print("\n📊 DETECTION SUMMARY:")
    print(results_df['decision'].value_counts())
    print(f"\n📈 Average Confidence: {results_df['confidence'].mean():.3f}")
    print(f"🎯 High Confidence (>0.8): {(results_df['confidence'] > 0.8).sum()} reviews")
    
    return results_df

print("Processing function ready!")

Processing function ready!


# *[IV] Main Execution*

In [10]:
df = pd.read_csv('../1_SIMPLE_CLEAN/simple_clean.csv')

In [11]:
results_df = process_dataset('../1_SIMPLE_CLEAN/simple_clean.csv', use_llm=True)

# Save results
results_df.to_csv('indo_fake_review.csv', index=False)
print("\n✅ Results saved to 'indo_fake_review.csv'")

# Show sample results
print("\n📋 Sample Results:")
print(results_df[['text', 'decision', 'confidence', 'suspicion_score']].head(10))


Processing 7875 reviews...
Mode: LLM ENABLED (slow)
Expected time: 1575.0 minutes

Loading LLM model (one-time initialization)...



Device set to use cpu


LLM loaded!
✓ Processed 100/7875 | Speed: 2.8 reviews/sec | ETA: 45.5 min
✓ Processed 200/7875 | Speed: 3.1 reviews/sec | ETA: 41.4 min
✓ Processed 300/7875 | Speed: 3.4 reviews/sec | ETA: 37.5 min
✓ Processed 400/7875 | Speed: 3.5 reviews/sec | ETA: 35.2 min
✓ Processed 500/7875 | Speed: 3.7 reviews/sec | ETA: 33.6 min
✓ Processed 600/7875 | Speed: 4.1 reviews/sec | ETA: 29.3 min
✓ Processed 700/7875 | Speed: 4.5 reviews/sec | ETA: 26.7 min
✓ Processed 800/7875 | Speed: 5.0 reviews/sec | ETA: 23.8 min
✓ Processed 900/7875 | Speed: 5.4 reviews/sec | ETA: 21.6 min
  💾 Checkpoint saved: 1000 reviews
✓ Processed 1000/7875 | Speed: 5.7 reviews/sec | ETA: 20.1 min
✓ Processed 1100/7875 | Speed: 5.8 reviews/sec | ETA: 19.6 min
✓ Processed 1200/7875 | Speed: 6.0 reviews/sec | ETA: 18.6 min
✓ Processed 1300/7875 | Speed: 6.1 reviews/sec | ETA: 18.0 min
✓ Processed 1400/7875 | Speed: 6.3 reviews/sec | ETA: 17.0 min
✓ Processed 1500/7875 | Speed: 6.4 reviews/sec | ETA: 16.6 min
✓ Processed 1600/

In [12]:
df

,ulasan_cleaned,Rating,Membantu
0,barang pesanan datang sesuai jadwal barang pes...,5,0
1,bahan: bagus manfaat produk: sangat baik jenis...,5,0
2,bahan: bagus manfaat produk: mencerahkan jenis...,5,1
3,bahan: sesuai pemesanan manfaat produk: mencer...,5,0
4,bahan: bagus sdah berlangganan manfaat produk:...,5,0
...,...,...,...
7870,lens clarity: good,5,0
7871,sesuai harga👌🏻,5,0
7872,nicee,5,0
7873,"sesuai harga, dan cocok untuk setup minimalis",5,0


In [13]:
print(df.ulasan_cleaned[5])

print(results_df.text[5])

barang sampai dgn mulus, udah pakai ini 2mingguan wajah keliatan sedikit cerah. semoga cocok terus diaku, trmksh🤍
barang sampai dgn mulus, udah pakai ini 2mingguan wajah keliatan sedikit cerah. semoga cocok terus d...
